# 2 — AlloyDB Graph

**AWS-side equivalent:** Aurora Serverless v2 + pgvector
**GCP service:** AlloyDB for PostgreSQL 16 + pgvector

AWS-side notebook: `notebooks/aws/02-aurora-graph.ipynb`

> **Why AlloyDB and not Cloud SQL PG?** AlloyDB is the GA-recommended primary
> for vector + relational workloads on GCP: pgvector is pre-installed, the
> ScaNN index ships out of the box (faster than IVFFlat at scale), and the
> AlloyDB Language Connector handles mTLS + IAM auth without a sidecar. Cloud
> SQL PG 16 also supports pgvector, but lacks ScaNN — fine for the workshop
> if you want a cheaper alternative. Set `cfg['gcp']['db_kind'] = 'cloudsql'`
> to take that path; cells below default to AlloyDB.

## Steps

1. Create an AlloyDB cluster + primary instance via `gcloud alloydb`
2. Connect through the AlloyDB Language Connector (mTLS + IAM, no sidecar)
3. Apply `schema/orgiq_schema.sql` (relational org graph)
4. Insert people / teams / skills from `data/org/*.json`
5. Verify with a small SQL query (CTE traversal — Aurora does NOT have Apache AGE)
6. Track resources via `ResourceTracker` for cleanup notebook

## Step 1: Load config + decide cluster names

In [ ]:
import json
from pathlib import Path
from cleanup_utils import ResourceTracker

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.setdefault('gcp', {})

PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

# Resource names follow the orgiq-* convention (matches AWS-side).
CLUSTER_ID = gcp.setdefault('alloydb_cluster', 'orgiq-alloydb')
INSTANCE_ID = gcp.setdefault('alloydb_instance', 'orgiq-alloydb-primary')
DB_NAME = gcp.setdefault('alloydb_db', 'orgiq')
DB_USER = gcp.setdefault('alloydb_user', 'postgres')
DB_PASSWORD_SECRET = gcp.setdefault('alloydb_password_secret', 'orgiq-alloydb-password')
NETWORK = gcp.setdefault('alloydb_network', 'default')

print(f'project   : {PROJECT_ID}')
print(f'region    : {REGION}')
print(f'cluster   : {CLUSTER_ID}')
print(f'instance  : {INSTANCE_ID}')
print(f'database  : {DB_NAME}')

tracker = ResourceTracker()

## Step 2: Stash the DB password in Secret Manager (one-time)

We never hard-code passwords. The cell below creates a Secret Manager
secret if it doesn't exist and prompts the user to seed it once. The AlloyDB
connector will fetch the value at connection time.

In [ ]:
from google.cloud import secretmanager
from google.api_core.exceptions import AlreadyExists, NotFound
import secrets as _secrets

sm = secretmanager.SecretManagerServiceClient()
parent = f'projects/{PROJECT_ID}'
secret_path = f'{parent}/secrets/{DB_PASSWORD_SECRET}'

# Create the secret container if missing.
try:
    sm.create_secret(
        request={
            'parent': parent,
            'secret_id': DB_PASSWORD_SECRET,
            'secret': {'replication': {'automatic': {}}},
        }
    )
    print(f'✓ Created secret container: {DB_PASSWORD_SECRET}')
except AlreadyExists:
    print(f'  Secret container already exists: {DB_PASSWORD_SECRET}')

# Add an initial version if none exists.
try:
    sm.access_secret_version(request={'name': f'{secret_path}/versions/latest'})
    print('✓ Secret has at least one version')
except NotFound:
    initial = _secrets.token_urlsafe(24)
    sm.add_secret_version(
        request={'parent': secret_path, 'payload': {'data': initial.encode()}}
    )
    print('✓ Seeded secret with a new random password (24 chars urlsafe)')

# Read it back for use in the cluster-create call.
DB_PASSWORD = sm.access_secret_version(
    request={'name': f'{secret_path}/versions/latest'}
).payload.data.decode()
print(f'✓ Password resolved (len={len(DB_PASSWORD)})')

## Step 3: Provision AlloyDB cluster + primary instance

Uses `gcloud alloydb` via subprocess. The Python SDK
(`google.cloud.alloydb_v1.AlloyDBAdminClient`) works too; we use the CLI
because it handles the multi-step LRO wait cleanly. Both routes are
idempotent — re-running re-uses the existing cluster.

**Cost note:** AlloyDB minimum primary is 2 vCPUs (`--cpu-count=2`). Idle
cost ~$0.36/hr (vs ~$0.06/hr for Cloud SQL PG db-f1-micro). Tear down via
notebook 19 when done.

In [ ]:
import subprocess

def _gcloud(args: list[str]) -> subprocess.CompletedProcess:
    cmd = ['gcloud', *args, '--project', PROJECT_ID, '--quiet']
    print('  $', ' '.join(cmd))
    return subprocess.run(cmd, capture_output=True, text=True)

# Does the cluster already exist?
describe = _gcloud(['alloydb', 'clusters', 'describe', CLUSTER_ID, '--region', REGION])
if describe.returncode == 0:
    print(f'✓ Cluster {CLUSTER_ID} already exists — skipping create.')
else:
    create = _gcloud([
        'alloydb', 'clusters', 'create', CLUSTER_ID,
        '--region', REGION,
        '--password', DB_PASSWORD,
        '--database-version', 'POSTGRES_16',
        '--network', NETWORK,
    ])
    if create.returncode != 0:
        raise RuntimeError(f'Cluster create failed:\nSTDOUT: {create.stdout}\nSTDERR: {create.stderr}')
    print(f'✓ Cluster {CLUSTER_ID} created')

tracker.track('alloydb_cluster', CLUSTER_ID, region=REGION)
tracker.flush()

# Primary instance.
describe_inst = _gcloud([
    'alloydb', 'instances', 'describe', INSTANCE_ID,
    '--cluster', CLUSTER_ID, '--region', REGION,
])
if describe_inst.returncode == 0:
    print(f'✓ Instance {INSTANCE_ID} already exists — skipping create.')
else:
    create_inst = _gcloud([
        'alloydb', 'instances', 'create', INSTANCE_ID,
        '--cluster', CLUSTER_ID,
        '--region', REGION,
        '--instance-type', 'PRIMARY',
        '--cpu-count', '2',
        # Public IP is convenient for notebook dev. For production, drop this
        # flag and connect via Private Service Connect or the VPC peering you
        # attached when creating the cluster.
        '--assign-inbound-public-ip', 'ASSIGN_IPV4',
    ])
    if create_inst.returncode != 0:
        raise RuntimeError(f'Instance create failed:\nSTDOUT: {create_inst.stdout}\nSTDERR: {create_inst.stderr}')
    print(f'✓ Instance {INSTANCE_ID} created')

INSTANCE_URI = f'projects/{PROJECT_ID}/locations/{REGION}/clusters/{CLUSTER_ID}/instances/{INSTANCE_ID}'
gcp['alloydb_instance_uri'] = INSTANCE_URI
config_path.write_text(json.dumps(config, indent=2))
print(f'✓ Instance URI: {INSTANCE_URI}')

## Step 4: Connect via the AlloyDB Language Connector

The connector handles mTLS, IAM, and IP refresh transparently. Pattern is
the same as Cloud SQL's connector. We use `psycopg` (v3) here because the
rest of OrgIQ assumes async-capable psycopg.

Install (in requirements.txt): `google-cloud-alloydb-connector[psycopg]`,
`psycopg[binary]>=3.2`.

In [ ]:
from google.cloud.alloydbconnector import Connector
import psycopg

# One connector per process. Reuse across cells.
_connector = Connector()

def get_conn() -> psycopg.Connection:
    """Open a new psycopg connection through the AlloyDB connector."""
    return _connector.connect(
        INSTANCE_URI,
        'psycopg',
        user=DB_USER,
        password=DB_PASSWORD,
        db=DB_NAME,
        ip_type='PUBLIC',  # switch to 'PRIVATE' if your instance is private-only
    )

# The default `postgres` DB exists on every AlloyDB cluster; the `orgiq` DB
# is created in the next cell if it isn't there yet.
def _ensure_database(db_name: str) -> None:
    bootstrap = _connector.connect(
        INSTANCE_URI, 'psycopg',
        user=DB_USER, password=DB_PASSWORD, db='postgres', ip_type='PUBLIC',
    )
    bootstrap.autocommit = True
    with bootstrap.cursor() as cur:
        cur.execute('SELECT 1 FROM pg_database WHERE datname = %s', (db_name,))
        if cur.fetchone() is None:
            cur.execute(f'CREATE DATABASE "{db_name}"')
            print(f'✓ Created database {db_name!r}')
        else:
            print(f'  Database {db_name!r} already exists')
    bootstrap.close()

_ensure_database(DB_NAME)

# Sanity check.
with get_conn() as conn, conn.cursor() as cur:
    cur.execute('SELECT current_database(), version()')
    db, ver = cur.fetchone()
    print(f'✓ Connected to {db} — {ver.split(",")[0]}')

## Step 5: Enable pgvector and apply the OrgIQ schema

AlloyDB ships pgvector pre-installed — `CREATE EXTENSION` is still required
per-database. The schema file is shared with the AWS side and lives at
`../../schema/orgiq_schema.sql`.

In [ ]:
SCHEMA_PATH = Path('../../schema/orgiq_schema.sql')
assert SCHEMA_PATH.exists(), f'Schema file missing: {SCHEMA_PATH.resolve()}'
schema_sql = SCHEMA_PATH.read_text()

# Note: schema/orgiq_schema.sql declares `vector(1024)` (Titan dim). For GCP
# defaults (gemini-embedding-001 @ 1536d), we patch the dim BEFORE applying.
GCP_EMBED_DIM = gcp.get('embedding_dim', 1536)
schema_sql = schema_sql.replace('vector(1024)', f'vector({GCP_EMBED_DIM})')

with get_conn() as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute('CREATE EXTENSION IF NOT EXISTS vector')
        cur.execute(schema_sql)
print(f'✓ pgvector enabled, schema applied (embedding dim={GCP_EMBED_DIM})')

## Step 6: Insert people / teams / skills from `data/org/`

Shared seed JSON with the AWS notebook. Uses `INSERT ... ON CONFLICT DO
UPDATE` for idempotent re-runs (matches the OrgIQ convention).

In [ ]:
people = json.loads(Path('data/org/people.json').read_text())
teams = json.loads(Path('data/org/teams.json').read_text())
skills_taxonomy = json.loads(Path('data/org/skills-taxonomy.json').read_text())

# Flatten skill taxonomy → [(id, name, domain)]. Shape is
#   {'categories': [{'name': <domain>, 'skills': [{'id', 'name', 'subcategory'}, ...]}]}
skill_rows = []
if isinstance(skills_taxonomy, dict) and 'categories' in skills_taxonomy:
    for cat in skills_taxonomy['categories']:
        domain = cat.get('name')
        for s in cat.get('skills', []):
            skill_rows.append({'id': s['id'], 'name': s['name'], 'domain': domain})
elif isinstance(skills_taxonomy, list):
    # Already a flat list of {id, name, domain} records.
    skill_rows = skills_taxonomy
else:
    raise ValueError(f'Unrecognized skills-taxonomy shape: {type(skills_taxonomy)}')

print(f'  people: {len(people)}  teams: {len(teams)}  skills: {len(skill_rows)}')

In [ ]:
with get_conn() as conn, conn.cursor() as cur:
    # Teams first (people.team is a free-text department, so we mint team rows
    # from the teams.json file directly).
    for t in teams:
        cur.execute(
            """
            INSERT INTO teams (id, name, parent_id)
            VALUES (%(id)s, %(name)s, %(parent_id)s)
            ON CONFLICT (id) DO UPDATE
              SET name = EXCLUDED.name,
                  parent_id = EXCLUDED.parent_id
            """,
            {'id': t['id'], 'name': t['name'], 'parent_id': t.get('parent_id')},
        )

    for s in skill_rows:
        cur.execute(
            """
            INSERT INTO skills (id, name, domain)
            VALUES (%(id)s, %(name)s, %(domain)s)
            ON CONFLICT (id) DO UPDATE
              SET name = EXCLUDED.name,
                  domain = EXCLUDED.domain
            """,
            s,
        )

    for p in people:
        cur.execute(
            """
            INSERT INTO people (id, name, email, title, team)
            VALUES (%(id)s, %(name)s, %(email)s, %(title)s, %(team)s)
            ON CONFLICT (id) DO UPDATE
              SET name = EXCLUDED.name,
                  email = EXCLUDED.email,
                  title = EXCLUDED.title,
                  team = EXCLUDED.team,
                  updated_at = NOW()
            """,
            {
                'id': p['id'],
                'name': p['name'],
                'email': p.get('email'),
                'title': p.get('title'),
                'team': p.get('department') or p.get('team'),
            },
        )
    conn.commit()
print(f'✓ Upserted {len(teams)} teams, {len(skill_rows)} skills, {len(people)} people')

## Step 7: Verify with a small graph traversal (pure SQL)

AWS-side uses Apache AGE Cypher; AlloyDB (like Aurora PG) does **not** have
AGE, so we model edges as join tables and traverse them with recursive
CTEs / multi-join SELECTs. Same data shape — different syntax.

In [ ]:
with get_conn() as conn, conn.cursor() as cur:
    for label, table in [('people', 'people'), ('teams', 'teams'), ('skills', 'skills')]:
        cur.execute(f'SELECT COUNT(*) FROM {table}')
        print(f'  {label:8s}: {cur.fetchone()[0]} rows')

    # Example traversal: people in the Platform Engineering dept, grouped by
    # title. (Real KNOWS_ABOUT edges land via the synth pipeline; this is the
    # one shape we can verify from raw seed data.)
    cur.execute(
        """
        SELECT title, COUNT(*) AS n
        FROM people
        WHERE team = 'Platform Engineering'
        GROUP BY title
        ORDER BY n DESC
        """
    )
    print('\nPlatform Engineering breakdown:')
    for title, n in cur.fetchall():
        print(f'  {title:30s} {n}')

## Step 8: Persist connection info to `config.json`

In [ ]:
gcp['alloydb_cluster'] = CLUSTER_ID
gcp['alloydb_instance'] = INSTANCE_ID
gcp['alloydb_instance_uri'] = INSTANCE_URI
gcp['alloydb_db'] = DB_NAME
gcp['alloydb_user'] = DB_USER
gcp['alloydb_password_secret'] = DB_PASSWORD_SECRET
config_path.write_text(json.dumps(config, indent=2))
print('✓ config.json updated:')
for k in ('alloydb_cluster', 'alloydb_instance', 'alloydb_db', 'alloydb_password_secret'):
    print(f'  gcp.{k} = {gcp[k]}')

# Don't leave the connector open across notebook restarts.
_connector.close()

## Summary

| Resource | Details |
|---|---|
| **DB engine** | AlloyDB for PostgreSQL 16 (pgvector pre-installed) |
| **Connection** | AlloyDB Language Connector (mTLS + IAM, no sidecar) |
| **Idle cost** | ~$0.36/hr (2 vCPU primary, min) |
| **Schema** | `../../schema/orgiq_schema.sql`, vector dim patched to GCP default |
| **Seed** | `data/org/{people,teams,skills-taxonomy}.json` |
| **Graph queries** | Pure SQL with edge tables (no Apache AGE) |

**Next:** `03-embeddings.ipynb` — generate Vertex AI embeddings for each
person and skill, write them to `node_embeddings`.